# SU(2) Pareto Circuit Sampling

This notebook takes one Hamiltonian target and samples a cloud of candidate circuits across several fixed CZ templates, including 2-, 3-, 4-, and 5-CZ line templates. It then refines each candidate and reports the tradeoff between unitary fidelity and simple hardware-cost proxies such as CZ count, local-gate count, and how far refinement moved the local `SU(2)` gates.

In [ ]:
BRANCH = "codex/circuit-diversity-diagnostic"
!pip install -q --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}


In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    ParetoScoringConfig,
    center_names_for_config,
    get_experiment_config,
    make_hamiltonian_target,
    plot_circuit_diversity,
    plot_pareto_circuit_sampling,
    print_circuit_diversity,
    print_circuit_diversity_summary,
    print_hamiltonian_target,
    print_pareto_circuit_candidates,
    print_pareto_circuit_summary,
    rescore_pareto_circuit_result,
    run_circuit_diversity_diagnostic,
    run_experiment,
    run_pareto_circuit_sampling,
)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)


## Train the local SU(2) gate generator

This is the same conditional Clifford-neighborhood generator used in the Hamiltonian demo. The Pareto cells below reuse its generated local gates.

In [ ]:
CONFIG_NAME = "baseline-clifford-cond"
EVAL_COUNT = 1000

config = get_experiment_config(CONFIG_NAME)
config = replace(config, sample_count=EVAL_COUNT, reference_count=EVAL_COUNT)

result = run_experiment(config, device=device)

center_names = center_names_for_config(result.config.data)
local_gates = result.generated_stochastic
local_labels = [center_names[int(label)] for label in result.stochastic_labels]

print(f"trained config: {result.config.name}")
print(f"generated local gates: {tuple(local_gates.shape)}")

## Pareto sampling on a transverse-Ising-style target

The score is post-hoc: `regularized score = refined fidelity - hardware cost`. It does not change the diffusion training loss or the refinement objective.

In [ ]:
ising_target = make_hamiltonian_target(
    [
        ("ZZI", 0.35),
        ("IZZ", 0.35),
        ("XII", 0.20),
        ("IXI", 0.20),
        ("IIX", 0.20),
    ],
    time=0.8,
    name="three-qubit-transverse-ising",
    n_qubits=3,
    device=device,
)

pareto_scoring = ParetoScoringConfig(
    cz_weight=0.015,
    local_gate_weight=0.0005,
    movement_weight=0.04,
    angle_weight=0.001,
)

ising_pareto = run_pareto_circuit_sampling(
    ising_target,
    generated_gates=local_gates,
    generated_labels=local_labels,
    templates=(
        "line-2cz-a",
        "line-2cz-b",
        "line-3cz-a",
        "line-3cz-b",
        "all-3cz",
        "line-4cz",
        "line-4cz-b",
        "line-5cz-a",
        "line-5cz-b",
    ),
    n_random_candidates=3000,
    top_k_per_template=5,
    refinement_steps=60,
    refinement_lr=0.05,
    threshold=0.99,
    scoring=pareto_scoring,
    seed=12007,
    show_progress=True,
)

print_hamiltonian_target(ising_target)
print()
print_pareto_circuit_candidates(ising_pareto, max_rows=12)
print()
print_pareto_circuit_summary(ising_pareto)
plot_pareto_circuit_sampling(ising_pareto)

## Fixed-template diversity diagnostic

This cell isolates the diversity question: for one Hamiltonian and one template, do many selected generated proposals refine to genuinely distinct circuits, or do they collapse into one neighborhood of `SU(2)^n`? Increase `n_selected` after the smoke run if you want a denser distribution.


In [ ]:
ising_diversity = run_circuit_diversity_diagnostic(
    ising_target,
    generated_gates=local_gates,
    generated_labels=local_labels,
    template="line-4cz",
    source="generated-search",
    n_random_candidates=10_000,
    n_selected=40,
    refinement_steps=60,
    refinement_lr=0.05,
    threshold=0.99,
    cluster_radius=0.15,
    seed=13007,
    show_progress=True,
)

print_circuit_diversity(ising_diversity, max_rows=12)
print()
print_circuit_diversity_summary(ising_diversity)
plot_circuit_diversity(ising_diversity)


## Optional: change the hardware-cost weights without rerunning search

This cell rescales the score to penalize CZ gates more heavily, using the same already-refined candidate cloud.

In [ ]:
heavier_cz_scoring = replace(pareto_scoring, cz_weight=0.03)
ising_pareto_heavier_cz = rescore_pareto_circuit_result(ising_pareto, heavier_cz_scoring)

print_pareto_circuit_candidates(ising_pareto_heavier_cz, max_rows=12)
print()
print_pareto_circuit_summary(ising_pareto_heavier_cz)
plot_pareto_circuit_sampling(ising_pareto_heavier_cz)